# LIMPIEZA Y UNIFICACIÓN DE DATOS

El presente cuaderno limpia, estandariza y unifica los registros electorales históricos del partido Nuevo Liberalismo en Colombia. El cuaderno hace parte de un ejercicio exploratorio, analítico y prescriptivo, en el que, a través de la estadística,

Los datasets provienen de fuentes oficiales. Los registros electorales son descargados de manera individual en las páginas habilitadas por la Resgistraduría, en su mayoría como archivos ``.csv``, que son estandarizados, limpiados y unificados para su posterior uso en ejercicios analíticos como la realización de análisis estadístico exploratorio, (EDA), diseño de tableros de control, ejecución de consultas SQL (duckdb sin modelo relacional completo) y su uso para la prescripción de acciones y recomendaciones para la segmentación, fortalecimiento y mejora del rendimiento del partido de cara a futuras elecciones. 

Se toman como referencia dos periodos de relevancia: 

1) 1980-1988: Constituye el periodo inicial e histórico del partido. En este, el Nuevo Liberalismo se consolida de movimiento a partido político, obteniendo su primera personería jurídica hasta 1988, año en el que Luis Carlos Galán se reintegra al Partido Liberal y renuncia a la personería jurídica del partido. 

2) 2021-2026: Representa el periodo de renovación del partido, marcado por la devolución de la personería jurídica en el 2021. 

## Bibliotecas

In [1]:
# Carga y manejo de archivos
from pathlib import Path
import os
import glob
import csv

# Procesamiento y limpieza de datos
import pandas as pd
import numpy as np
import re
from unidecode import unidecode

# Opcional: detección de encoding si algún CSV falla al leer
import chardet
from charset_normalizer import from_path

# Opcional: manejo de fechas
from datetime import datetime

# Opcional: warnings
import warnings
warnings.filterwarnings("ignore")

## Limpieza y unificación parcial

Se opta por un enfoque de limpieza de datos que prioriza la eficiencia y la claridad. Se utilizan bibliotecas estándar de Python como pandas y numpy para manipular los datos, mientras que se emplean expresiones regulares y la biblioteca unidecode para normalizar los textos. Además, se implementa un manejo cuidadoso de los archivos y directorios mediante pathlib y os, asegurando que el código sea robusto y fácil de mantener.

Además, se elige una limpieza de dos fases teniendo en cuenta la calidad y cantidad de archivos. Por ejemplo, los datos de elecciones como congreso 2026 y 2022 se hayan repartidos en varios archivos ``.csv`` o ``.xlsx``, separados por departamentos, lo cual añade pasos adicionales a su limpieza y unificación. Por ende, una vez unificados los archivos, se puede pasar a la construcción de la base de datos final.

### 1980-1988

In [2]:
from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# Ruta de la carpeta del bloque 1980-1988
ruta_carpeta = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\1980-1988")

# Buscar todos los archivos CSV
archivos_csv = sorted(ruta_carpeta.glob("*.csv"))

# Validación básica
if not archivos_csv:
    raise FileNotFoundError(f"No se encontraron archivos CSV en: {ruta_carpeta}")

# Cargar todos los archivos
dfs = []
dfs_por_nombre = {}

for archivo in archivos_csv:
    try:
        df_temp = pd.read_csv(archivo, low_memory=False, encoding="utf-8")
    except UnicodeDecodeError:
        try:
            df_temp = pd.read_csv(archivo, low_memory=False, encoding="latin-1")
        except UnicodeDecodeError:
            df_temp = pd.read_csv(archivo, low_memory=False, encoding="cp1252")

    # Agregar metadato útil
    df_temp["archivo_origen"] = archivo.name

    # Guardar en lista
    dfs.append(df_temp)

    # Guardar en diccionario con nombre limpio
    nombre_df = archivo.stem.lower().replace(" ", "_").replace("-", "_")
    dfs_por_nombre[nombre_df] = df_temp

# Confirmación
print(f"Archivos cargados: {len(dfs)}")
print("Nombres disponibles en dfs_por_nombre:")
for nombre in dfs_por_nombre.keys():
    print("-", nombre)

Archivos cargados: 16
Nombres disponibles en dfs_por_nombre:
- 1980_asamblea.dta
- 1980_concejo.dta
- 1982_asamblea.dta
- 1982_camara.dta
- 1982_concejo.dta
- 1982_presidencia.dta
- 1982_senado.dta
- 1984_asamblea.dta
- 1984_concejo.dta
- 1986_asamblea.dta
- 1986_camara.dta
- 1986_concejo.dta
- 1986_presidencia.dta
- 1986_senado.dta
- 1988_alcaldia.dta
- 1988_asamblea.dta


In [4]:
# Comparación de columnas entre dataframes
columnas_por_archivo = {
    nombre: set(df_temp.columns)
    for nombre, df_temp in dfs_por_nombre.items()
}

# Unión total de columnas
union_columnas = sorted(set().union(*columnas_por_archivo.values()))

# Columnas comunes a todos los archivos
columnas_comunes = sorted(set.intersection(*columnas_por_archivo.values()))

# Columnas exclusivas por archivo
columnas_exclusivas = {
    nombre: sorted(cols - set(columnas_comunes))
    for nombre, cols in columnas_por_archivo.items()
}

# Resumen por archivo
df_columnas = pd.DataFrame([
    {
        "archivo": nombre,
        "n_columnas": len(cols),
        "columnas_exclusivas": ", ".join(sorted(cols - set(columnas_comunes)))
    }
    for nombre, cols in columnas_por_archivo.items()
]).sort_values("archivo")

print("UNIÓN TOTAL DE COLUMNAS:")
print(union_columnas)

print("\nCOLUMNAS COMUNES A TODOS LOS ARCHIVOS:")
print(columnas_comunes)

print("\nRESUMEN POR ARCHIVO:")
print(df_columnas)

UNIÓN TOTAL DE COLUMNAS:
['ano', 'archivo_origen', 'circunscripcion', 'coddpto', 'coddpto80', 'coddpto82', 'coddpto84', 'coddpto86', 'coddpto88', 'codigo_lista', 'codigo_partido', 'codmpio', 'codmpio80', 'codmpio82', 'codmpio84', 'codmpio86', 'codmpio88', 'curules', 'departamento', 'departamento80', 'departamento82', 'departamento84', 'departamento86', 'departamento88', 'en_lista', 'fecha_eleccion', 'id_electoral', 'municipio', 'nombres', 'primer_apellido', 'segundo_apellido', 'tipo_eleccion', 'votos']

COLUMNAS COMUNES A TODOS LOS ARCHIVOS:
['ano', 'archivo_origen', 'circunscripcion', 'coddpto', 'codigo_lista', 'codigo_partido', 'codmpio', 'curules', 'departamento', 'fecha_eleccion', 'id_electoral', 'municipio', 'nombres', 'primer_apellido', 'segundo_apellido', 'tipo_eleccion', 'votos']

RESUMEN POR ARCHIVO:
                 archivo  n_columnas  \
0      1980_asamblea.dta          21   
1       1980_concejo.dta          21   
2      1982_asamblea.dta          21   
3        1982_camar

In [ ]:
import pandas as pd
from pathlib import Path

def estandarizar_columnas_segura(df):
    df = df.copy()

    renombres = {
        "coddpto80": "coddpto",
        "coddpto82": "coddpto",
        "coddpto84": "coddpto",
        "coddpto86": "coddpto",
        "coddpto88": "coddpto",
        "codmpio80": "codmpio",
        "codmpio82": "codmpio",
        "codmpio84": "codmpio",
        "codmpio86": "codmpio",
        "codmpio88": "codmpio",
        "departamento80": "departamento",
        "departamento82": "departamento",
        "departamento84": "departamento",
        "departamento86": "departamento",
        "departamento88": "departamento"
    }

    for col_old, col_new in renombres.items():
        if col_old in df.columns:
            if col_new in df.columns:
                df[col_new] = df[col_new].where(df[col_new].notna(), df[col_old])
                df = df.drop(columns=[col_old])
            else:
                df = df.rename(columns={col_old: col_new})

    if "en_lista" not in df.columns:
        df["en_lista"] = pd.NA

    return df

# Aplica esto sobre tu lista de dataframes ya cargada: dfs
dfs_limpios = [estandarizar_columnas_segura(df_temp) for df_temp in dfs]

# Concatenar
df_bloque_1980_1988 = pd.concat(dfs_limpios, ignore_index=True, sort=False)

# Normalizar nombres de columnas
df_bloque_1980_1988.columns = [c.lower() for c in df_bloque_1980_1988.columns]

# Exportar
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "bloque_1980_1988_maestro.csv"
df_bloque_1980_1988.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Filas:", df_bloque_1980_1988.shape[0])
print("Columnas:", df_bloque_1980_1988.shape[1])
print("Archivo guardado en:", csv_path)

Filas: 534169
Columnas: 18
Archivo guardado en: output\bloque_1980_1988_maestro.csv


In [13]:
df_1980_1988 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\bloque_1980_1988_maestro.csv")
df_1980_1988.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534169 entries, 0 to 534168
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id_electoral      534169 non-null  int64  
 1   ano               534169 non-null  int64  
 2   tipo_eleccion     534169 non-null  object 
 3   fecha_eleccion    534169 non-null  object 
 4   coddpto           534169 non-null  float64
 5   departamento      534151 non-null  object 
 6   codmpio           534169 non-null  float64
 7   municipio         534067 non-null  object 
 8   circunscripcion   534169 non-null  object 
 9   codigo_partido    502402 non-null  float64
 10  codigo_lista      534169 non-null  int64  
 11  en_lista          248999 non-null  float64
 12  primer_apellido   291646 non-null  object 
 13  segundo_apellido  243537 non-null  object 
 14  nombres           501992 non-null  object 
 15  votos             534166 non-null  float64
 16  curules           49

In [14]:
import pandas as pd
import unicodedata

def quitar_tildes(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = unicodedata.normalize("NFKD", x)
    return "".join(c for c in x if not unicodedata.combining(c))

# 1) Eliminar fecha_eleccion
df_1980_1988 = df_1980_1988.drop(columns=["fecha_eleccion","codigo_lista","en_lista","curules","archivo_origen"], errors="ignore")

# 2) Homogeneizar texto y quitar tildes
text_cols = df_1980_1988.select_dtypes(include=["object", "string"]).columns
df_1980_1988[text_cols] = (
    df_1980_1988[text_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.lower().map(quitar_tildes))
    .fillna("sin_dato")
)

# 3) Ajustar dtypes numéricos
df_1980_1988[["id_electoral", "ano", "coddpto", "codmpio", "codigo_partido"]] = (
    df_1980_1988[["id_electoral", "ano", "coddpto", "codmpio", "codigo_partido"]]
    .astype("Int64")
)

# 4) Votos como entero nullable
df_1980_1988["votos"] = pd.to_numeric(df_1980_1988["votos"], errors="coerce").round().astype("Int64")

# 5) Revisión
df_1980_1988.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534169 entries, 0 to 534168
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id_electoral      534169 non-null  Int64 
 1   ano               534169 non-null  Int64 
 2   tipo_eleccion     534169 non-null  object
 3   coddpto           534169 non-null  Int64 
 4   departamento      534169 non-null  object
 5   codmpio           534169 non-null  Int64 
 6   municipio         534169 non-null  object
 7   circunscripcion   534169 non-null  object
 8   codigo_partido    502402 non-null  Int64 
 9   primer_apellido   534169 non-null  object
 10  segundo_apellido  534169 non-null  object
 11  nombres           534169 non-null  object
 12  votos             534166 non-null  Int64 
dtypes: Int64(6), object(7)
memory usage: 56.0+ MB


In [21]:
df_1980_1988["nombre_completo"] = (
    df_1980_1988["nombres"].fillna("sin_dato").str.strip() + " " +
    df_1980_1988["primer_apellido"].fillna("sin_dato").str.strip() + " " +
    df_1980_1988["segundo_apellido"].fillna("sin_dato").str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

In [31]:
reemplazos = {
    "movimiento nuevo liberalismo sin_dato sin_dato": "movimiento nuevo liberalismo",
    "luis carlos galan sin_dato": "luis carlos galan sarmiento",
    "jorge eliecer maldonado sin_dato": "jorge eliecer maldonado",
    "cecilia castro sin_dato": "cecilia castro",
    "herman lozano sin_dato": "herman lozano",
    "pedro julio sanchez sin_dato": "pedro julio sanchez",
    "partido nuevo liberalismo sin_dato sin_dato": "partido nuevo liberalismo",
    "movimiento de izquierda nuevo liberal 82 sin_dato sin_dato": "movimiento de izquierda nuevo liberal 82"
}

df_1980_1988["nombre_completo"] = df_1980_1988["nombre_completo"].replace(reemplazos)

In [46]:
import duckdb

query_sql = """
SELECT *
FROM 
    df_1980_1988
WHERE
    codigo_partido = '19790001'
    OR nombres IN ('movimiento nuevo liberalismo', 'movimiento de izquierda nuevo liberal 82','partido nuevo liberalismo')

"""

# Ejecutar la consulta y convertir el resultado a un nuevo DataFrame
df_88_sql = duckdb.sql(query_sql).df()
df_88_sql.head(100)

,id_electoral,ano,tipo_eleccion,coddpto,departamento,codmpio,municipio,circunscripcion,codigo_partido,primer_apellido,segundo_apellido,nombres,votos,nombre_completo
0,319820007,1982,camara de representantes,19,cauca,19001,popayan,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,1411,movimiento de izquierda nuevo liberal 82
1,319820007,1982,camara de representantes,19,cauca,19001,popayan,departamental,19790001,velasco,ramirez,omar henry,0,omar henry velasco ramirez
2,319820007,1982,camara de representantes,19,cauca,19022,almaguer,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,5,movimiento de izquierda nuevo liberal 82
3,319820007,1982,camara de representantes,19,cauca,19022,almaguer,departamental,19790001,velasco,ramirez,omar henry,0,omar henry velasco ramirez
4,319820007,1982,camara de representantes,19,cauca,19050,argelia,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,108,movimiento de izquierda nuevo liberal 82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,319820015,1982,camara de representantes,50,meta,50568,puerto gaitan,departamental,19790001,lopez,bejarano,jesus,0,jesus lopez bejarano
96,319820015,1982,camara de representantes,50,meta,50573,puerto lopez,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,243,movimiento de izquierda nuevo liberal 82
97,319820015,1982,camara de representantes,50,meta,50573,puerto lopez,departamental,19790001,lopez,bejarano,jesus,0,jesus lopez bejarano
98,319820015,1982,camara de representantes,50,meta,50577,puerto lleras,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,342,movimiento de izquierda nuevo liberal 82


In [47]:
df_88_sql.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14142 entries, 0 to 14141
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_electoral      14142 non-null  int64 
 1   ano               14142 non-null  int64 
 2   tipo_eleccion     14142 non-null  object
 3   coddpto           14142 non-null  int64 
 4   departamento      14142 non-null  object
 5   codmpio           14142 non-null  int64 
 6   municipio         14142 non-null  object
 7   circunscripcion   14142 non-null  object
 8   codigo_partido    14142 non-null  int64 
 9   primer_apellido   14142 non-null  object
 10  segundo_apellido  14142 non-null  object
 11  nombres           14142 non-null  object
 12  votos             14142 non-null  int64 
 13  nombre_completo   14142 non-null  object
dtypes: int64(6), object(8)
memory usage: 1.5+ MB


In [40]:
df_1980_1988.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\80_88_limpio.csv", index=False, encoding="utf-8-sig")

In [48]:
df_88_sql.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\80_88_nuevo_liberalismo.csv", index=False, encoding="utf-8-sig")

### 2021-2026

#### 2026/congreso

In [2]:
from pathlib import Path
import pandas as pd

carpeta = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad\2026")
salida = carpeta / "parciales"
salida.mkdir(exist_ok=True)

archivos = sorted(carpeta.glob("*.xlsx"))

for i, archivo in enumerate(archivos, 1):
    print(f"[{i}/{len(archivos)}] Leyendo: {archivo.name}")
    try:
        df = pd.read_excel(archivo)
        df.to_csv(salida / f"{archivo.stem}.csv", index=False, encoding="utf-8-sig")
        print(f"    OK -> {archivo.stem}.csv")
    except Exception as e:
        print(f"    ERROR -> {archivo.name}: {e}")

parciales = sorted(salida.glob("*.csv"))
df_congreso_2026 = pd.concat(
    [pd.read_csv(archivo, encoding="utf-8-sig") for archivo in parciales],
    ignore_index=True
)

[1/51] Leyendo: amazonas.xlsx
    OK -> amazonas.csv
[2/51] Leyendo: antioquia.xlsx
    OK -> antioquia.csv
[3/51] Leyendo: MMV_XXX_01_000_XXX_XX_XX_XXX_1036.csv.xlsx
    OK -> MMV_XXX_01_000_XXX_XX_XX_XXX_1036.csv.csv
[4/51] Leyendo: MMV_XXX_01_000_XXX_XX_XX_XXX_1039.csv.xlsx
    OK -> MMV_XXX_01_000_XXX_XX_XX_XXX_1039.csv.csv
[5/51] Leyendo: MMV_XXX_01_000_XXX_XX_XX_XXX_1046.csv.xlsx
    OK -> MMV_XXX_01_000_XXX_XX_XX_XXX_1046.csv.csv
[6/51] Leyendo: MMV_XXX_01_000_XXX_XX_XX_XXX_1049.csv.xlsx
    OK -> MMV_XXX_01_000_XXX_XX_XX_XXX_1049.csv.csv
[7/51] Leyendo: MMV_XXX_03_000_XXX_XX_XX_XXX_1001.csv.xlsx
    OK -> MMV_XXX_03_000_XXX_XX_XX_XXX_1001.csv.csv
[8/51] Leyendo: MMV_XXX_05_000_XXX_XX_XX_XXX_1002.csv.xlsx
    OK -> MMV_XXX_05_000_XXX_XX_XX_XXX_1002.csv.csv
[9/51] Leyendo: MMV_XXX_05_000_XXX_XX_XX_XXX_1041.csv.xlsx
    OK -> MMV_XXX_05_000_XXX_XX_XX_XXX_1041.csv.csv
[10/51] Leyendo: MMV_XXX_07_000_XXX_XX_XX_XXX_1003.csv.xlsx
    OK -> MMV_XXX_07_000_XXX_XX_XX_XXX_1003.csv.csv
[11

In [4]:
df_congreso_2026.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15242312 entries, 0 to 15242311
Data columns (total 19 columns):
 #   Column      Dtype  
---  ------      -----  
 0   DEP         int64  
 1   DEPNOMBRE   object 
 2   MUN         int64  
 3   MUNNOMBRE   object 
 4   ZONA        int64  
 5   PUESTO      object 
 6   PUESNOMBRE  object 
 7   MESA        int64  
 8   COMUCODIGO  int64  
 9   COMUNOMBRE  object 
 10  CORCODIGO   int64  
 11  CORNOMBRE   object 
 12  CIR         int64  
 13  PAR         int64  
 14  PARNOMBRE   object 
 15  CAN         int64  
 16  CANCEDULA   float64
 17  CANNOMBRE   object 
 18  VOTOS       int64  
dtypes: float64(1), int64(10), object(8)
memory usage: 2.2+ GB


In [6]:
df_congreso_2026["PUESTO"] = df_congreso_2026["PUESTO"].astype("string")

df_congreso_2026.to_parquet(
    salida / "df_congreso_2026.parquet",
    index=False
)

In [7]:
salida = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad")
salida.mkdir(exist_ok=True)

df_congreso_2026.to_csv(
    salida / "df_congreso_2026.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

In [13]:
import duckdb

query = """
SELECT *
FROM 
    df_congreso_2026
WHERE CANNOMBRE = 'JUAN MANUEL GALAN PACHON'
    OR PARNOMBRE LIKE '%NUEVO LIBERALISMO%'
"""

# Ejecutar la consulta y convertir el resultado a un nuevo DataFrame
df_26_sql = duckdb.sql(query).df()
df_26_sql.head(100)

,DEP,DEPNOMBRE,MUN,MUNNOMBRE,ZONA,PUESTO,PUESNOMBRE,MESA,COMUCODIGO,COMUNOMBRE,CORCODIGO,CORNOMBRE,CIR,PAR,PARNOMBRE,CAN,CANCEDULA,CANNOMBRE,VOTOS
0,60,AMAZONAS,1,LETICIA,98,1,CARCEL,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,3
1,60,AMAZONAS,1,LETICIA,99,77,KILOMETRO 11,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,2
2,60,AMAZONAS,1,LETICIA,99,76,IE SAN JUAN BOSCO,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,5
3,60,AMAZONAS,1,LETICIA,2,3,IE SAGRADO CORAZON DE JESUS,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,3
4,60,AMAZONAS,1,LETICIA,2,3,IE SAGRADO CORAZON DE JESUS,2,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,60,AMAZONAS,1,LETICIA,1,2,JORGE ELIECER GAITAN,8,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,1
96,60,AMAZONAS,1,LETICIA,1,2,JORGE ELIECER GAITAN,9,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,1
97,60,AMAZONAS,1,LETICIA,1,2,JORGE ELIECER GAITAN,10,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,2
98,60,AMAZONAS,1,LETICIA,1,2,JORGE ELIECER GAITAN,11,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,3


In [17]:
df_26_sql.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\26_congreso_nuevo_liberalismo.csv", index=False, encoding="utf-8-sig")

#### 2023/territoriales

In [19]:
df_territoriales_2023 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad\MMV_2023_NACIONAL.csv")
df_territoriales_2023.head()

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Código Zona,Código Puesto,Nombre Puesto,Mesa,Código Comuna,Nombre Comuna,Código Corporación,Nombre Corporación,Código Circunscripción,Código Partido,Nombre Partido,Código Candidato,Nombre Candidato,Total Votos
0,1,ANTIOQUIA,1,MEDELLIN,1,1,SEC. ESC. LA ESPERANZA NO 2,1,1,COMUNA 1 POPULAR,1,GOBERNADOR,1,0,CANDIDATOS TOTALES,997,VOTOS NULOS,5
1,1,ANTIOQUIA,1,MEDELLIN,1,1,SEC. ESC. LA ESPERANZA NO 2,1,1,COMUNA 1 POPULAR,1,GOBERNADOR,1,0,CANDIDATOS TOTALES,996,VOTOS EN BLANCO,2
2,1,ANTIOQUIA,1,MEDELLIN,1,1,SEC. ESC. LA ESPERANZA NO 2,1,1,COMUNA 1 POPULAR,1,GOBERNADOR,1,0,CANDIDATOS TOTALES,998,VOTOS NO MARCADOS,30
3,1,ANTIOQUIA,1,MEDELLIN,1,1,SEC. ESC. LA ESPERANZA NO 2,1,1,COMUNA 1 POPULAR,1,GOBERNADOR,1,2,PARTIDO CONSERVADOR COLOMBIANO,9,JUAN DIEGO GOMEZ JIMENEZ,2
4,1,ANTIOQUIA,1,MEDELLIN,1,1,SEC. ESC. LA ESPERANZA NO 2,1,1,COMUNA 1 POPULAR,1,GOBERNADOR,1,24,PARTIDO DEMÓCRATA COLOMBIANO,2,JULIAN BEDOYA PULGARIN,1


In [23]:
query = """
SELECT *
FROM 
    df_territoriales_2023
WHERE "Nombre Partido" LIKE '%NUEVO LIBERALISMO%'
"""

# Ejecutar la consulta y convertir el resultado a un nuevo DataFrame
df_23_sql = duckdb.sql(query).df()
df_23_sql.head(100)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Código Zona,Código Puesto,Nombre Puesto,Mesa,Código Comuna,Nombre Comuna,Código Corporación,Nombre Corporación,Código Circunscripción,Código Partido,Nombre Partido,Código Candidato,Nombre Candidato,Total Votos
0,1,ANTIOQUIA,1,MEDELLIN,7,1,COL AGUSTINIANO DE SAN NICOLAS,1,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,82,PAOLA STEFANIA GUISAO URREGO,1
1,1,ANTIOQUIA,1,MEDELLIN,7,1,COL AGUSTINIANO DE SAN NICOLAS,10,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
2,1,ANTIOQUIA,1,MEDELLIN,7,1,COL AGUSTINIANO DE SAN NICOLAS,11,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,2
3,1,ANTIOQUIA,1,MEDELLIN,7,1,COL AGUSTINIANO DE SAN NICOLAS,11,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,81,JONATHAN ANDRES ALVAREZ MACHADO,2
4,1,ANTIOQUIA,1,MEDELLIN,7,1,COL AGUSTINIANO DE SAN NICOLAS,13,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,ANTIOQUIA,1,MEDELLIN,7,3,I.E. GILBERTO ALZATE AVENDAÑO,10,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,82,PAOLA STEFANIA GUISAO URREGO,1
96,1,ANTIOQUIA,1,MEDELLIN,7,3,I.E. GILBERTO ALZATE AVENDAÑO,12,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
97,1,ANTIOQUIA,1,MEDELLIN,7,3,I.E. GILBERTO ALZATE AVENDAÑO,12,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,81,JONATHAN ANDRES ALVAREZ MACHADO,1
98,1,ANTIOQUIA,1,MEDELLIN,7,3,I.E. GILBERTO ALZATE AVENDAÑO,12,4,COMUNA 4 ARANJUEZ,5,JAL,3,19,PARTIDO NUEVO LIBERALISMO,82,PAOLA STEFANIA GUISAO URREGO,1


In [24]:
df_23_sql.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\23_territoriales_nuevo_liberalismo.csv", index=False, encoding="utf-8-sig")

#### 2022/congreso

In [29]:
import pandas as pd
from pathlib import Path

carpeta = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad\2022")
archivos = sorted(carpeta.glob("MMV_2022_*.csv"))

encodings = ["utf-16-le", "utf-8", "cp1252", "latin1"]
separador = ","

dfs = []

for archivo in archivos:
    df = None
    for enc in encodings:
        try:
            df = pd.read_csv(
                archivo,
                encoding=enc,
                sep=separador,
                engine="python",
                on_bad_lines="skip"
            )
            print(f"{archivo.name} -> {enc}, {df.shape}")
            break
        except Exception as e:
            print(f"{archivo.name} falló con {enc}: {type(e).__name__}")
            continue
    if df is not None:
        dfs.append(df)
    else:
        print(f"{archivo.name} no se pudo leer con ninguna codificación")

df_congreso_2022 = pd.concat(dfs, ignore_index=True) if dfs else None
if df_congreso_2022 is not None:
    print("Shape final:", df_congreso_2022.shape)
    print("Columnas:", df_congreso_2022.columns.tolist())

MMV_2022_01_ANTIOQUIA.csv -> utf-16-le, (1346671, 21)
MMV_2022_03_ATLANTICO.csv -> utf-16-le, (498184, 21)
MMV_2022_05_BOLIVAR.csv -> utf-16-le, (363811, 21)
MMV_2022_07_BOYACA.csv -> utf-16-le, (258451, 21)
MMV_2022_09_CALDAS.csv -> utf-16-le, (199349, 21)
MMV_2022_11_CAUCA.csv -> utf-16-le, (191014, 21)
MMV_2022_12_CESAR.csv -> utf-16-le, (186559, 21)
MMV_2022_13_CORDOBA.csv -> utf-16-le, (227727, 21)
MMV_2022_15_CUNDINAMARCA.csv -> utf-16-le, (554780, 21)
MMV_2022_16_BOGOTA.csv -> utf-16-le, (1759916, 21)
MMV_2022_17_CHOCO.csv -> utf-16-le, (59529, 21)
MMV_2022_19_HUILA.csv -> utf-16-le, (192198, 21)
MMV_2022_21_MAGDALENA.csv -> utf-16-le, (219065, 21)
MMV_2022_23_NARIÑO.csv -> utf-16-le, (228868, 21)
MMV_2022_24_RISARALDA.csv -> utf-16-le, (176662, 21)
MMV_2022_25_NORTE_DE_SAN.csv -> utf-16-le, (293554, 21)
MMV_2022_26_QUINDIO.csv -> utf-16-le, (108217, 21)
MMV_2022_27_SANTANDER.csv -> utf-16-le, (446213, 21)
MMV_2022_28_SUCRE.csv -> utf-16-le, (155338, 21)
MMV_2022_29_TOLIMA.csv -

In [32]:
import pandas as pd
from pathlib import Path

carpeta = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad\2022")

# 1. Departamentos (todos excepto 88_CONSULADOS)
archivos_dept = sorted(
    p for p in carpeta.glob("MMV_2022_*.csv")
    if "88_CONSULADOS" not in p.name
)

dfs = []
for archivo in archivos_dept:
    df = pd.read_csv(
        archivo,
        encoding="utf-16-le",
        sep=",",
        engine="python",
        on_bad_lines="skip"
    )
    dfs.append(df)

# 2. Consulados
archivo_consulados = carpeta / "MMV_2022_88_CONSULADOS.csv"
df_consulados = pd.read_csv(
    archivo_consulados,
    encoding="utf-8-sig",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)
dfs.append(df_consulados)

# 3. Concatenar
df_congreso_2022 = pd.concat(dfs, ignore_index=True)

print("Shape final:", df_congreso_2022.shape)
print("Columnas:", df_congreso_2022.columns.tolist())

Shape final: (9325778, 21)
Columnas: ['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Código Zona', 'Código Puesto', 'Nombre Puesto', 'Mesa', 'Código Comuna', 'Nombre Comuna', 'Código Corporación', 'Nombre Corporación', 'Código Circunscripción', 'Nombre Circunscripción', 'Código CITREP', 'Nombre CITREP', 'Código Partido', 'Nombre Partido', 'Código Candidato', 'Nombre Candidato', 'Total Votos']


In [33]:
# Exportar a Parquet (recomendado: snappy compression)
df_congreso_2022.to_parquet(
    r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\2021-actualidad\df_congreso_2022.parquet",
    index=False,
    compression="snappy"
)

In [34]:
query = """
SELECT *
FROM 
    df_congreso_2022
WHERE "Nombre Partido" LIKE '%NUEVO LIBERALISMO%'
"""

# Ejecutar la consulta y convertir el resultado a un nuevo DataFrame
df_22_sql = duckdb.sql(query).df()
df_22_sql.head(100)

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Código Zona,Código Puesto,Nombre Puesto,Mesa,Código Comuna,Nombre Comuna,...,Nombre Corporación,Código Circunscripción,Nombre Circunscripción,Código CITREP,Nombre CITREP,Código Partido,Nombre Partido,Código Candidato,Nombre Candidato,Total Votos
0,1,ANTIOQUIA,49,BELLO,99,80,SAN FELIX,6,12.0,12COMUNA 12 SAN FELIX,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
1,1,ANTIOQUIA,49,BELLO,99,80,SAN FELIX,4,12.0,12COMUNA 12 SAN FELIX,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
2,1,ANTIOQUIA,49,BELLO,6,6,IE JOSEFA CAMPOS,8,7.0,07COMUNA 7 ALTOS DE NIQUIA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
3,1,ANTIOQUIA,49,BELLO,6,6,IE JOSEFA CAMPOS,9,7.0,07COMUNA 7 ALTOS DE NIQUIA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
4,1,ANTIOQUIA,49,BELLO,6,6,IE JOSEFA CAMPOS,12,7.0,07COMUNA 7 ALTOS DE NIQUIA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,ANTIOQUIA,49,BELLO,2,2,COLEGIO PARROQUIAL SAN FCO DE ASIS,19,2.0,02COMUNA 2 MADERA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
96,1,ANTIOQUIA,49,BELLO,2,2,COLEGIO PARROQUIAL SAN FCO DE ASIS,23,2.0,02COMUNA 2 MADERA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
97,1,ANTIOQUIA,49,BELLO,2,2,COLEGIO PARROQUIAL SAN FCO DE ASIS,27,2.0,02COMUNA 2 MADERA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1
98,1,ANTIOQUIA,49,BELLO,7,2,I.E FE Y ALEGRIA NUEVA GENERAC,1,8.0,08COMUNA 8 NIQUIA,...,SENADO,1,NACIONAL,0,NO CITREP,19,PARTIDO NUEVO LIBERALISMO,0,PARTIDO NUEVO LIBERALISMO,1


In [35]:
df_22_sql.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\22_congreso_nuevo_liberalismo.csv", index=False, encoding="utf-8-sig")

## Limpieza y unificación final

In [15]:
df1 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\Nuevo Liberalismo\80_88_nuevo_liberalismo.csv")
df2 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\Nuevo Liberalismo\22_congreso_nuevo_liberalismo.csv")
df3 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\Nuevo Liberalismo\23_territoriales_nuevo_liberalismo.csv")
df4 = pd.read_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\Nuevo Liberalismo\26_congreso_nuevo_liberalismo.csv")



In [19]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14142 entries, 0 to 14141
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_electoral      14142 non-null  int64 
 1   ano               14142 non-null  int64 
 2   tipo_eleccion     14142 non-null  object
 3   coddpto           14142 non-null  int64 
 4   departamento      14142 non-null  object
 5   codmpio           14142 non-null  int64 
 6   municipio         14142 non-null  object
 7   circunscripcion   14142 non-null  object
 8   codigo_partido    14142 non-null  int64 
 9   primer_apellido   14142 non-null  object
 10  segundo_apellido  14142 non-null  object
 11  nombres           14142 non-null  object
 12  votos             14142 non-null  int64 
 13  nombre_completo   14142 non-null  object
dtypes: int64(6), object(8)
memory usage: 1.5+ MB


In [16]:
df1.head()

,id_electoral,ano,tipo_eleccion,coddpto,departamento,codmpio,municipio,circunscripcion,codigo_partido,primer_apellido,segundo_apellido,nombres,votos,nombre_completo
0,319820007,1982,camara de representantes,19,cauca,19001,popayan,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,1411,movimiento de izquierda nuevo liberal 82
1,319820007,1982,camara de representantes,19,cauca,19001,popayan,departamental,19790001,velasco,ramirez,omar henry,0,omar henry velasco ramirez
2,319820007,1982,camara de representantes,19,cauca,19022,almaguer,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,5,movimiento de izquierda nuevo liberal 82
3,319820007,1982,camara de representantes,19,cauca,19022,almaguer,departamental,19790001,velasco,ramirez,omar henry,0,omar henry velasco ramirez
4,319820007,1982,camara de representantes,19,cauca,19050,argelia,departamental,19790001,sin_dato,sin_dato,movimiento de izquierda nuevo liberal 82,108,movimiento de izquierda nuevo liberal 82


In [21]:
import pandas as pd

# =========================================
# 1. Copiar df1 para no modificar el original
# =========================================

df1_limpio = df1.copy()

# =========================================
# 2. Eliminar columnas innecesarias
# =========================================

cols_a_eliminar = [
    'id_electoral',
    'circunscripcion',
    'codigo_partido',
    'primer_apellido',
    'segundo_apellido',
    'nombres'
]

df1_limpio = df1_limpio.drop(columns=cols_a_eliminar, errors='ignore')

# =========================================
# 3. Reemplazar valores en 'nombre_completo' ANTES de renombrar
# =========================================

reemplazos = {
    'movimiento de izquierda nuevo liberal 82': 'lista',
    'partido nuevo liberalismo larrarte rodriguez': 'larrarte rodriguez',
    'movimiento nuevo liberalismo': 'lista',
    'partido nuevo liberalismo quijano caballero': 'quijano caballero',
    'partido nuevo liberalismo': 'lista'
}

# Aplicar reemplazos (case-insensitive)
df1_limpio['nombre_completo'] = df1_limpio['nombre_completo'].str.lower().replace(reemplazos)

# =========================================
# 4. Renombrar columnas
# =========================================

df1_limpio = df1_limpio.rename(
    columns={
        'nombre_completo': 'candidato',
        'tipo_eleccion': 'corporacion'
    }
)

# =========================================
# 5. Verificar resultado
# =========================================

print("=== df1_limpio ===")
print("Shape:", df1_limpio.shape)
print("Columnas:", df1_limpio.columns.tolist())
print("\nValores únicos en 'candidato' después de limpiar:")
print(df1_limpio['candidato'].value_counts(dropna=False).head(20))

=== df1_limpio ===
Shape: (14142, 8)
Columnas: ['ano', 'corporacion', 'coddpto', 'departamento', 'codmpio', 'municipio', 'votos', 'candidato']

Valores únicos en 'candidato' después de limpiar:
candidato
lista                              6374
luis carlos galan sarmiento        1195
gabriel jaime norena henao          246
hernando aguilera blanco            230
carlos ardila ballesteros           170
luis barreto buitrago               140
german riano cano                   139
silvio de jesus mejia duque         125
pedro jaramillo monsalve            124
luis guillermo valencia jimenez     122
luis ivan marulanda gomez           122
alberto uribe mejia                 122
ivan de jesus restrepo gomez        122
jorge e londono ulloa               121
jorge eliecer maldonado             121
roberto sandoval ballesteros        121
jose corredor nunez                 117
rafael amador campos                117
alberto villamizar cardenas         117
cesar pardo villalba                

In [30]:
# Añadir columna 'partido' con valor 'nuevo liberalismo' en todos los registros
df1_limpio['partido'] = 'nuevo liberalismo'

# Verificar
print("=== df1_limpio ===")
print("Shape:", df1_limpio.shape)
print("Columnas:", df1_limpio.columns.tolist())
print("\nValores únicos en 'partido':")
print(df1_limpio['partido'].value_counts(dropna=False))

=== df1_limpio ===
Shape: (14142, 9)
Columnas: ['ano', 'corporacion', 'coddpto', 'departamento', 'codmpio', 'municipio', 'votos', 'candidato', 'partido']

Valores únicos en 'partido':
partido
nuevo liberalismo    14142
Name: count, dtype: int64


In [27]:
df2_limpio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188414 entries, 0 to 188413
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   coddpto       188414 non-null  int64 
 1   departamento  188414 non-null  object
 2   codmpio       188414 non-null  int64 
 3   municipio     188414 non-null  object
 4   corporacion   188414 non-null  object
 5   candidato     188414 non-null  object
 6   votos         188414 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 10.1+ MB


In [29]:
import pandas as pd

# =========================================
# 1. Copiar df2 para no modificar el original
# =========================================

df2_limpio = df2.copy()

# =========================================
# 2. Seleccionar solo las columnas equivalentes
# =========================================

cols_a_mantener = [
    'Código Departamento',
    'Nombre Departamento',
    'Código Municipio',
    'Nombre Municipio',
    'Nombre Corporación',
    'Nombre Candidato',
    'Total Votos'
]

df2_limpio = df2_limpio[cols_a_mantener]

# =========================================
# 3. Renombrar a esquema df1
# =========================================

mapa_columnas = {
    'Código Departamento': 'coddpto',
    'Nombre Departamento': 'departamento',
    'Código Municipio': 'codmpio',
    'Nombre Municipio': 'municipio',
    'Nombre Corporación': 'corporacion',
    'Nombre Candidato': 'candidato',
    'Total Votos': 'votos'
}

df2_limpio = df2_limpio.rename(columns=mapa_columnas)

# =========================================
# 4. Añadir columna 'partido' con valor 'nuevo liberalismo'
# =========================================

df2_limpio['partido'] = 'nuevo liberalismo'

# =========================================
# 5. Añadir columna 'ano' con valor 2022
# =========================================

df2_limpio['ano'] = 2022

# =========================================
# 6. Reemplazar 'PARTIDO NUEVO LIBERALISMO' por 'lista' en 'candidato'
# =========================================

df2_limpio['candidato'] = df2_limpio['candidato'].str.replace(
    'PARTIDO NUEVO LIBERALISMO',
    'lista',
    regex=False
)

# =========================================
# 7. Poner en minúscula y sin tildes todo el contenido
# =========================================

def quitar_tildes(texto):
    if isinstance(texto, str):
        reemplazos = str.maketrans(
            'áéíóúüñÁÉÍÓÚÜÑ',
            'aeiouunAEIOUUN'
        )
        return texto.translate(reemplazos).lower()
    return texto

# Aplicar a todas las columnas de texto
cols_texto = df2_limpio.select_dtypes(include=['object']).columns

for col in cols_texto:
    df2_limpio[col] = df2_limpio[col].apply(quitar_tildes)

# =========================================
# 8. Verificar resultado
# =========================================

print("=== df2_limpio ===")
print("Shape:", df2_limpio.shape)
print("Columnas:", df2_limpio.columns.tolist())
print("\nPrimeras 5 filas:")
print(df2_limpio.head())
print("\nValores únicos en 'candidato':")
print(df2_limpio['candidato'].value_counts(dropna=False).head(20))

=== df2_limpio ===
Shape: (188414, 9)
Columnas: ['coddpto', 'departamento', 'codmpio', 'municipio', 'corporacion', 'candidato', 'votos', 'partido', 'ano']

Primeras 5 filas:
   coddpto departamento  codmpio municipio corporacion candidato  votos  \
0        1    antioquia       49     bello      senado     lista      1   
1        1    antioquia       49     bello      senado     lista      1   
2        1    antioquia       49     bello      senado     lista      1   
3        1    antioquia       49     bello      senado     lista      1   
4        1    antioquia       49     bello      senado     lista      1   

             partido   ano  
0  nuevo liberalismo  2022  
1  nuevo liberalismo  2022  
2  nuevo liberalismo  2022  
3  nuevo liberalismo  2022  
4  nuevo liberalismo  2022  

Valores únicos en 'candidato':
candidato
lista                                      106210
julia miranda londono                       12171
miguel andres silva moyano                   8117
german ri

In [32]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695629 entries, 0 to 695628
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Código Departamento     695629 non-null  int64 
 1   Nombre Departamento     695629 non-null  object
 2   Código Municipio        695629 non-null  int64 
 3   Nombre Municipio        695629 non-null  object
 4   Código Zona             695629 non-null  int64 
 5   Código Puesto           695629 non-null  object
 6   Nombre Puesto           695629 non-null  object
 7   Mesa                    695629 non-null  int64 
 8   Código Comuna           695629 non-null  int64 
 9   Nombre Comuna           695629 non-null  object
 10  Código Corporación      695629 non-null  int64 
 11  Nombre Corporación      695629 non-null  object
 12  Código Circunscripción  695629 non-null  int64 
 13  Código Partido          695629 non-null  int64 
 14  Nombre Partido          695629 non-n

In [34]:
import pandas as pd

# =========================================
# 1. Copiar df3 para no modificar el original
# =========================================

df3_limpio = df3.copy()

# =========================================
# 2. Seleccionar solo las columnas equivalentes
# =========================================

cols_a_mantener = [
    'Código Departamento',
    'Nombre Departamento',
    'Código Municipio',
    'Nombre Municipio',
    'Nombre Corporación',
    'Nombre Candidato',
    'Nombre Partido',
    'Total Votos'
]

df3_limpio = df3_limpio[cols_a_mantener]

# =========================================
# 3. Renombrar a esquema df3
# =========================================

mapa_columnas = {
    'Código Departamento': 'coddpto',
    'Nombre Departamento': 'departamento',
    'Código Municipio': 'codmpio',
    'Nombre Municipio': 'municipio',
    'Nombre Corporación': 'corporacion',  # Si quieres guardar también el nombre
    'Nombre Candidato': 'candidato',
    'Nombre Partido': 'partido',
    'Total Votos': 'votos'
}

df3_limpio = df3_limpio.rename(columns=mapa_columnas)



# =========================================
# 4. Verificar resultado
# =========================================

print("=== df3_limpio ===")
print("Shape:", df3_limpio.shape)
print("Columnas:", df3_limpio.columns.tolist())
print("\nPrimeras 5 filas:")
print(df3_limpio.head())

=== df3_limpio ===
Shape: (695629, 8)
Columnas: ['coddpto', 'departamento', 'codmpio', 'municipio', 'corporacion', 'candidato', 'partido', 'votos']

Primeras 5 filas:
   coddpto departamento  codmpio municipio corporacion  \
0        1    ANTIOQUIA        1  MEDELLIN         JAL   
1        1    ANTIOQUIA        1  MEDELLIN         JAL   
2        1    ANTIOQUIA        1  MEDELLIN         JAL   
3        1    ANTIOQUIA        1  MEDELLIN         JAL   
4        1    ANTIOQUIA        1  MEDELLIN         JAL   

                         candidato                    partido  votos  
0     PAOLA STEFANIA GUISAO URREGO  PARTIDO NUEVO LIBERALISMO      1  
1        PARTIDO NUEVO LIBERALISMO  PARTIDO NUEVO LIBERALISMO      1  
2        PARTIDO NUEVO LIBERALISMO  PARTIDO NUEVO LIBERALISMO      2  
3  JONATHAN ANDRES ALVAREZ MACHADO  PARTIDO NUEVO LIBERALISMO      2  
4        PARTIDO NUEVO LIBERALISMO  PARTIDO NUEVO LIBERALISMO      3  


In [36]:
import pandas as pd

# =========================================
# 1. Convertir todos los valores en minúscula y sin tildes
# =========================================

def quitar_tildes(texto):
    if isinstance(texto, str):
        reemplazos = str.maketrans(
            'áéíóúüñÁÉÍÓÚÜÑ',
            'aeiouunAEIOUUN'
        )
        return texto.translate(reemplazos).lower()
    return texto

# Aplicar a todas las columnas de texto
cols_texto = df3_limpio.select_dtypes(include=['object']).columns

for col in cols_texto:
    df3_limpio[col] = df3_limpio[col].apply(quitar_tildes)

# =========================================
# 2. Cambiar "PARTIDO NUEVO LIBERALISMO" por 'lista' en 'candidato'
# =========================================

df3_limpio['candidato'] = df3_limpio['candidato'].str.replace(
    'partido nuevo liberalismo',
    'lista',
    regex=False
)

# =========================================
# 3. Crear columna 'ano' con valor 2023
# =========================================

df3_limpio['ano'] = 2023

# =========================================
# 4. Verificar resultado
# =========================================

print("=== df3_limpio ===")
print("Shape:", df3_limpio.shape)
print("Columnas:", df3_limpio.columns.tolist())
print("\nValores únicos en 'candidato':")
print(df3_limpio['candidato'].value_counts(dropna=False).head(20))

=== df3_limpio ===
Shape: (695629, 9)
Columnas: ['coddpto', 'departamento', 'codmpio', 'municipio', 'corporacion', 'candidato', 'partido', 'votos', 'ano']

Valores únicos en 'candidato':
candidato
lista                                               41284
nuevo liberalismo en marcha                         18988
carlos fernando galan pachon                        17784
juan javier baena merlano                           14382
juan manuel diaz martinez                           10785
nuevo liberalismo- agrupacion politica en marcha    10185
ricardo andres correa mojica                         8316
jesus david araque mejia                             8163
cristina calderon restrepo                           7644
juan david quintero rubio                            7560
fernando lopez gutierrez                             7254
david hernando saavedra murcia                       6366
elkin jwiseb huertas carrasquilla                    5964
juan enrique aaron rivero                        

In [40]:
# Lista de valores a reemplazar por 'lista'
valores_a_reemplazar = [
    'nuevo liberalismo- agrupacion politica en marcha',
    'nuevo liberalismo - conservador -colombia just...',
    'partido de la u-nuevo liberalismo-colombia jus...',
    'nuevo liberalismo-nueva fuerza democratica',
    'u, nuevo liberalismo, mira',
    'coalicion nuevo liberalismo en marcha',
    'nuevo liberalismo - cambio radical',
    'nuevo liberalismo y mira al concejo',
    'coalicion nuevo liberalismo - dignidad y compr...',
    'coalicion nuevo liberalismo y mira',
    'colombia renaciente - nuevo liberalismo',
    'nuevo liberalismo en marcha',
    'nuevo liberalismo - mira',
    'dignidad & compromiso y nuevo liberalismo',
    'alianza verde nuevo liberalismo'
]

# Reemplazar todos estos valores por 'lista' en la columna 'candidato'
df3_limpio.loc[df3_limpio['candidato'].isin(valores_a_reemplazar), 'candidato'] = 'lista'

# Verificar
print("Valores únicos en 'candidato' después del reemplazo:")
print(df3_limpio['candidato'].value_counts(dropna=False).head(20))

Valores únicos en 'candidato' después del reemplazo:
candidato
lista                                  74105
carlos fernando galan pachon           17784
juan javier baena merlano              14382
juan manuel diaz martinez              10785
ricardo andres correa mojica            8316
jesus david araque mejia                8163
cristina calderon restrepo              7644
juan david quintero rubio               7560
fernando lopez gutierrez                7254
david hernando saavedra murcia          6366
elkin jwiseb huertas carrasquilla       5964
juan enrique aaron rivero               5465
julian fernando silva cala              5344
andres mauricio prieto sanchez          4836
adriana marcela gutierrez castaneda     4805
yesid fernando rivera contreras         4566
evert rengifo hernandez                 4529
flor elva cardenas velandia             4486
cesar augusto salamanca rojas           4483
jose ignacio gutierrez bolivar          4358
Name: count, dtype: int64


In [43]:
df4.head()

,DEP,DEPNOMBRE,MUN,MUNNOMBRE,ZONA,PUESTO,PUESNOMBRE,MESA,COMUCODIGO,COMUNOMBRE,CORCODIGO,CORNOMBRE,CIR,PAR,PARNOMBRE,CAN,CANCEDULA,CANNOMBRE,VOTOS
0,60,AMAZONAS,1,LETICIA,98,1,CARCEL,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,3
1,60,AMAZONAS,1,LETICIA,99,77,KILOMETRO 11,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,2
2,60,AMAZONAS,1,LETICIA,99,76,IE SAN JUAN BOSCO,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,5
3,60,AMAZONAS,1,LETICIA,2,3,IE SAGRADO CORAZON DE JESUS,1,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,3
4,60,AMAZONAS,1,LETICIA,2,3,IE SAGRADO CORAZON DE JESUS,2,0,NACIONAL,6,CONSULTAS,0,200,LA GRAN CONSULTA POR COLOMBIA,4,79589617.0,JUAN MANUEL GALAN PACHON,10


In [42]:
df4.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['DEP', 'DEPNOMBRE', 'MUN', 'MUNNOMBRE', 'ZONA', 'PUESTO', 'PUESNOMBRE',
       'MESA', 'COMUCODIGO', 'COMUNOMBRE', 'CORCODIGO', 'CORNOMBRE', 'CIR',
       'PAR', 'PARNOMBRE', 'CAN', 'CANCEDULA', 'CANNOMBRE', 'VOTOS'],
      dtype='object')>

In [45]:
df4_limpio = df4.copy()

# =========================================
# 2. Seleccionar solo las columnas equivalentes
# =========================================

cols_a_mantener = [
    'DEP',
    'DEPNOMBRE',
    'MUN',
    'MUNNOMBRE',
    'CORNOMBRE',
    'CANNOMBRE',
    'PARNOMBRE',
    'VOTOS'
]

df4_limpio = df4_limpio[cols_a_mantener]

# =========================================
# 3. Renombrar a esquema df4
# =========================================

mapa_columnas = {
    'DEP': 'coddpto',
    'DEPNOMBRE': 'departamento',
    'MUN': 'codmpio',
    'MUNNOMBRE': 'municipio',
    'CORNOMBRE': 'corporacion',  # Si quieres guardar también el nombre
    'CANNOMBRE': 'candidato',
    'PARNOMBRE': 'partido',
    'VOTOS': 'votos'
}

df4_limpio = df4_limpio.rename(columns=mapa_columnas)



# =========================================
# 4. Verificar resultado
# =========================================

print("=== df4_limpio ===")
print("Shape:", df4_limpio.shape)
print("Columnas:", df4_limpio.columns.tolist())
print("\nPrimeras 5 filas:")
print(df4_limpio.head())


=== df4_limpio ===
Shape: (194179, 8)
Columnas: ['coddpto', 'departamento', 'codmpio', 'municipio', 'corporacion', 'candidato', 'partido', 'votos']

Primeras 5 filas:
   coddpto departamento  codmpio municipio corporacion  \
0       60     AMAZONAS        1   LETICIA   CONSULTAS   
1       60     AMAZONAS        1   LETICIA   CONSULTAS   
2       60     AMAZONAS        1   LETICIA   CONSULTAS   
3       60     AMAZONAS        1   LETICIA   CONSULTAS   
4       60     AMAZONAS        1   LETICIA   CONSULTAS   

                  candidato                        partido  votos  
0  JUAN MANUEL GALAN PACHON  LA GRAN CONSULTA POR COLOMBIA      3  
1  JUAN MANUEL GALAN PACHON  LA GRAN CONSULTA POR COLOMBIA      2  
2  JUAN MANUEL GALAN PACHON  LA GRAN CONSULTA POR COLOMBIA      5  
3  JUAN MANUEL GALAN PACHON  LA GRAN CONSULTA POR COLOMBIA      3  
4  JUAN MANUEL GALAN PACHON  LA GRAN CONSULTA POR COLOMBIA     10  


In [50]:
import pandas as pd

# =========================================
# 1. Minúsculas y quitar tildes
# =========================================

def quitar_tildes(texto):
    if isinstance(texto, str):
        reemplazos = str.maketrans(
            'áéíóúüñÁÉÍÓÚÜÑ',
            'aeiouunAEIOUUN'
        )
        return texto.translate(reemplazos).lower()
    return texto

# Aplicar a todas las columnas de texto
cols_texto = df4_limpio.select_dtypes(include=['object']).columns

for col in cols_texto:
    df4_limpio[col] = df4_limpio[col].apply(quitar_tildes)

# =========================================
# 2. Crear columna 'ano' con valor 2026
# =========================================

df4_limpio['ano'] = 2026

# =========================================
# 3. Reemplazar valores por 'lista' en 'candidato'
# =========================================

valores_a_reemplazar = [
    'cr-nuevo liberalismo',
    'partido nuevo liberalismo',
    'centro democratico- nuevo liberalismo-mira',
    'partido de la u, mira, nuevo liberalismo'
]

df4_limpio.loc[df4_limpio['candidato'].isin(valores_a_reemplazar), 'candidato'] = 'lista'

# =========================================
# 4. Verificar resultado
# =========================================

print("=== df4_limpio ===")
print("Shape:", df4_limpio.shape)
print("Columnas:", df4_limpio.columns.tolist())
print("\nValores únicos en 'candidato':")
print(df4_limpio['candidato'].value_counts(dropna=False).head(20))

=== df4_limpio ===
Shape: (194179, 9)
Columnas: ['coddpto', 'departamento', 'codmpio', 'municipio', 'corporacion', 'candidato', 'partido', 'votos', 'ano']

Valores únicos en 'candidato':
candidato
juan manuel galan pachon                       102634
lista                                           19248
juan carlos restrepo hoyos                      11768
maria de los angeles ardila hernandez            7240
centro democrãtico- nuevo liberalismo-mira      6070
fabian camilo rojas barrera                      5792
carlos alberto perez molina                      5772
hector josue camacho acosta                      4118
felipe de jesus zapata donado                    3542
daniel santos carrillo                           3052
nadia alexandra carreã‘o avella                  2828
vicente anibal ojeda martinez                    2816
vanessa diaz cepeda                              2734
oscar yovany montenegro salcedo                  2632
camilo andres astaiza puerta                   

In [51]:
# Reemplazo más flexible (contenga "centro democr" y "nuevo liberalismo")
mask = df4_limpio['candidato'].str.contains('centro democr', case=False, na=False)
df4_limpio.loc[mask, 'candidato'] = 'lista'

# Verificar
print(df4_limpio['candidato'].value_counts(dropna=False).head(20))

candidato
juan manuel galan pachon                 102634
lista                                     25318
juan carlos restrepo hoyos                11768
maria de los angeles ardila hernandez      7240
fabian camilo rojas barrera                5792
carlos alberto perez molina                5772
hector josue camacho acosta                4118
felipe de jesus zapata donado              3542
daniel santos carrillo                     3052
nadia alexandra carreã‘o avella            2828
vicente anibal ojeda martinez              2816
vanessa diaz cepeda                        2734
oscar yovany montenegro salcedo            2632
camilo andres astaiza puerta               2260
laura cristina monroy bayona               2208
mayra alejandra valencia balanta           1570
arifa del socorro rincon rodriguez         1556
julio cesar aguirre leon                   1421
jaime castilla verjel                       931
manuel felipe parra fletcher                764
Name: count, dtype: int64


In [56]:
df1_limpio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14142 entries, 0 to 14141
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ano           14142 non-null  int64 
 1   corporacion   14142 non-null  object
 2   coddpto       14142 non-null  int64 
 3   departamento  14142 non-null  object
 4   codmpio       14142 non-null  int64 
 5   municipio     14142 non-null  object
 6   votos         14142 non-null  int64 
 7   candidato     14142 non-null  object
 8   partido       14142 non-null  object
dtypes: int64(4), object(5)
memory usage: 994.5+ KB


In [57]:
import pandas as pd
from pathlib import Path

# =========================================
# 1. Concatenar los 4 dataframes
# =========================================

df_maestro = pd.concat(
    [df1_limpio, df2_limpio, df3_limpio, df4_limpio],
    ignore_index=True
)

print("Shape base maestra:", df_maestro.shape)
print("Columnas:", df_maestro.columns.tolist())

# =========================================
# 2. Crear carpeta si no existe
# =========================================

ruta_salida = Path(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\master")
ruta_salida.mkdir(parents=True, exist_ok=True)

# =========================================
# 3. Guardar como Parquet
# =========================================

archivo_parquet = ruta_salida / "df_maestro_elecciones_limpio.parquet"

df_maestro.to_parquet(
    archivo_parquet,
    index=False,
    compression="snappy"
)

print("\nBase maestra guardada como Parquet en:", archivo_parquet)
print("Shape final:", df_maestro.shape)

Shape base maestra: (1092364, 9)
Columnas: ['ano', 'corporacion', 'coddpto', 'departamento', 'codmpio', 'municipio', 'votos', 'candidato', 'partido']

Base maestra guardada como Parquet en: C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\data\master\df_maestro_elecciones_limpio.parquet
Shape final: (1092364, 9)


## Fase 3

In [58]:
df_maestro.head()

,ano,corporacion,coddpto,departamento,codmpio,municipio,votos,candidato,partido
0,1982,camara de representantes,19,cauca,19001,popayan,1411,lista,nuevo liberalismo
1,1982,camara de representantes,19,cauca,19001,popayan,0,omar henry velasco ramirez,nuevo liberalismo
2,1982,camara de representantes,19,cauca,19022,almaguer,5,lista,nuevo liberalismo
3,1982,camara de representantes,19,cauca,19022,almaguer,0,omar henry velasco ramirez,nuevo liberalismo
4,1982,camara de representantes,19,cauca,19050,argelia,108,lista,nuevo liberalismo


In [61]:
# Obtener valores únicos de la columna 'departamento' como lista de texto
valores_unicos_departamento = df_maestro['departamento'].unique().tolist()

print("Valores únicos en 'departamento':")
print(valores_unicos_departamento)
print(f"\nTotal de valores únicos: {len(valores_unicos_departamento)}")

Valores únicos en 'departamento':
['cauca', 'meta', 'antioquia', 'atlantico', 'consulados', 'bolivar', 'boyaca', 'caldas', 'caqueta', 'cesar', 'cordoba', 'bogota dc', 'cundinamarca', 'choco', 'huila', 'la guajira', 'magdalena', 'narino', 'norte de santander', 'quindio', 'risaralda', 'santander', 'sucre', 'tolima', 'valle del cauca', 'arauca', 'casanare', 'putumayo', 'archipielago de san andres providencia y santa catalina', 'amazonas', 'guainia', 'guaviare', 'vaupes', 'vichada', 'comisaria del vichada', 'sin_dato', 'san andres', 'santafe de bogota dc', 'bogota d.c.', 'norte de san', 'valle', 'nariã‘o']

Total de valores únicos: 42


In [62]:
import numpy as np

# =========================================
# 1. Diccionario de reemplazos para 'departamento'
# =========================================

reemplazos_departamento = {
    'bogota dc': 'bogota',
    'santafe de bogota dc': 'bogota',
    'bogota d.c.': 'bogota',
    'valle': 'valle del cauca',
    'nariã‘o': 'narino',
    'norte de san': 'norte de santander',
    'san andres': 'san andres, providencia y santa catalina',
    'comisaria del vichada': 'vichada'# o 'desconocido' si prefieres
}

# =========================================
# 2. Aplicar reemplazos
# =========================================

df_maestro['departamento'] = df_maestro['departamento'].replace(reemplazos_departamento)

# =========================================
# 3. Verificar resultado
# =========================================

valores_unicos_departamento = df_maestro['departamento'].unique().tolist()

print("Valores únicos en 'departamento' después de limpiar:")
print(sorted(valores_unicos_departamento))
print(f"\nTotal de valores únicos: {len(valores_unicos_departamento)}")

Valores únicos en 'departamento' después de limpiar:
['amazonas', 'antioquia', 'arauca', 'archipielago de san andres providencia y santa catalina', 'atlantico', 'bogota', 'bolivar', 'boyaca', 'caldas', 'caqueta', 'casanare', 'cauca', 'cesar', 'choco', 'consulados', 'cordoba', 'cundinamarca', 'guainia', 'guaviare', 'huila', 'la guajira', 'magdalena', 'meta', 'narino', 'norte de santander', 'putumayo', 'quindio', 'risaralda', 'san andres, providencia y santa catalina', 'santander', 'sin_dato', 'sucre', 'tolima', 'valle del cauca', 'vaupes', 'vichada']

Total de valores únicos: 36


In [63]:
# Reemplazar el valor restante
df_maestro.loc[df_maestro['departamento'] == 'archipielago de san andres providencia y santa catalina', 'departamento'] = 'san andres, providencia y santa catalina'

# Verificar
valores_unicos_departamento = df_maestro['departamento'].unique().tolist()

print("Valores únicos en 'departamento' después de limpiar:")
print(sorted(valores_unicos_departamento))
print(f"\nTotal de valores únicos: {len(valores_unicos_departamento)}")

Valores únicos en 'departamento' después de limpiar:
['amazonas', 'antioquia', 'arauca', 'atlantico', 'bogota', 'bolivar', 'boyaca', 'caldas', 'caqueta', 'casanare', 'cauca', 'cesar', 'choco', 'consulados', 'cordoba', 'cundinamarca', 'guainia', 'guaviare', 'huila', 'la guajira', 'magdalena', 'meta', 'narino', 'norte de santander', 'putumayo', 'quindio', 'risaralda', 'san andres, providencia y santa catalina', 'santander', 'sin_dato', 'sucre', 'tolima', 'valle del cauca', 'vaupes', 'vichada']

Total de valores únicos: 35


In [64]:
# Obtener valores únicos de 'municipio' ordenados
valores_unicos_municipio = sorted(df_maestro['municipio'].unique().tolist())

print("Valores únicos en 'municipio':")
for i, val in enumerate(valores_unicos_municipio, 1):
    print(f"{i}. {val}")

print(f"\nTotal de valores únicos: {len(valores_unicos_municipio)}")

Valores únicos en 'municipio':
1. abejorral
2. abrego
3. abriaqui
4. acacias
5. acandi
6. acevedo
7. achi
8. agrado
9. agua de dios
10. aguachica
11. aguada
12. aguadas
13. aguazul
14. agustin codazzi
15. aipe
16. alban
17. alban (san jose)
18. albania
19. alcala
20. aldana
21. alejandria
22. alemania
23. algarrobo
24. algeciras
25. almaguer
26. almeida
27. alpujarra
28. altamira
29. alto baudo
30. alto baudo (pie de pato)
31. altos del rosario
32. alvarado
33. amaga
34. amalfi
35. ambalema
36. anapoima
37. ancuya
38. andalucia
39. andes
40. angelopolis
41. angostura
42. anolaima
43. anori
44. anserma
45. ansermanuevo
46. antillas holandesas
47. antioquia
48. anza
49. anzoategui
50. apartado
51. apia
52. apulo
53. aquitania
54. aquitania (puebloviejo)
55. aracataca
56. aranzazu
57. aratoca
58. arauca
59. arauquita
60. arbelaez
61. arbeleez
62. arboleda
63. arboleda (berruecos)
64. arboledas
65. arboletes
66. arcabuco
67. arenal
68. argelia
69. argentina
70. ariguani
71. ariguani (el di

In [65]:
import numpy as np

# =========================================
# 1. Diccionario de reemplazos para 'municipio'
# =========================================

reemplazos_municipio = {
    # Caracteres extraños / ñ
    'brice?o': 'briceno',
    'briceã‘o': 'briceno',
    'ca?asgordas': 'canasgordas',
    'cove?as': 'covenas',
    'coveã‘as': 'covenas',
    'el pe?ol': 'el penol',
    'el peã‘ol': 'el penol',
    'el pe?on': 'el penon',
    'el peã‘on': 'el penon',
    'el pi?on': 'el pinon',
    'el piã‘on': 'el pinon',
    'la monta?ita': 'la montanita',
    'la montaã‘ita': 'la montanita',
    'la pe?a': 'la pena',
    'la peã‘a': 'la pena',
    'mo?itos': 'monitos',
    'moã‘itos': 'monitos',
    'nari?o': 'narino',
    'nariã‘o': 'narino',
    'oca?a': 'ocana',
    'ocaã‘a': 'ocana',
    'piji?o del carmen': 'pijino del carmen',
    'pijiã‘o del carmen': 'pijino del carmen',
    'puerto carre?o': 'puerto carreno',
    'puerto carreã‘o': 'puerto carreno',
    'puerto nari?o': 'puerto narino',
    'puerto nariã‘o': 'puerto narino',
    'salda?a': 'saldana',
    'saldaã‘a': 'saldana',
    'san jose de la monta?a': 'san jose de la montana',

    # Bogotá
    'bogota d e': 'bogota',
    'bogota dc': 'bogota',
    'bogota. d.c.': 'bogota',
    'santafe de bogota dc': 'bogota',

    # Paréntesis / cabeceras equivalentes
    'alban (san jose)': 'alban',
    'alto baudo (pie de pato)': 'alto baudo',
    'aquitania (puebloviejo)': 'aquitania',
    'arboleda (berruecos)': 'arboleda',
    'ariguani (el dificil)': 'ariguani',
    'armero (guayabal)': 'armero',
    'atrato (yuto)': 'atrato',
    'bahia solano (mutis)': 'bahia solano',
    'bajo baudo (pizarro)': 'bajo baudo',
    'bojaya (bellavista)': 'bojaya',
    'buenos aires (pacoa)': 'buenos aires',
    'calima (darien)': 'calima',
    'colon (genova)': 'colon',
    'coloso (ricaurte)': 'coloso',
    'cotorra (bongo)': 'cotorra',
    'cuaspud (carlosama)': 'cuaspud',
    'francisco pizarro (salahonda)': 'francisco pizarro',
    'galeras (nueva granada)': 'galeras',
    'la apartada (frontera)': 'la apartada',
    'la argentina (plata vieja)': 'la argentina',
    'lopez (micay)': 'lopez',
    'los andes (sotomayor)': 'los andes',
    'magui (payan)': 'magui',
    'mallama (piedrancha)': 'mallama',
    'medio atrato (bete)': 'medio atrato',
    'medio baudo (puerto meluk)': 'medio baudo',
    'paez (belalcazar)': 'paez',
    'paratebueno (la naguaya)': 'paratebueno',
    'patia (el bordo)': 'patia',
    'paz de ariporo (moreno)': 'paz de ariporo',
    'purace (coconuco)': 'purace',
    'roberto payan (san jose)': 'roberto payan',
    'san juan de betulia (betulia)': 'san juan de betulia',
    'san miguel (la dorada)': 'san miguel',
    'santa barbara (iscuande)': 'santa barbara',
    'santacruz (guachaves)': 'santacruz',
    'sotara (paispamba)': 'sotara',
    'tesalia (carnicerias)': 'tesalia',
    'zona bananera (sevilla)': 'zona bananera',

    # Duplicados o variantes
    'arbeleez': 'arbelaez',
    'calara': 'calarca',
    'don matias': 'donmatias',
    'cuaspud carlosama': 'cuaspud',
    'guadalajara de buga': 'buga',
    'guican de la sierra': 'guican',
    'manaure balcon del cesar (mana': 'manaure balcon del cesar',
    'palmas socorro': 'palmas del socorro',
    'pueblo rico': 'pueblorrico',
    'puerto nare (la magdalena)': 'puerto nare',
    'puerto nare-la magdalena': 'puerto nare',
    'san carlos guaroa': 'san carlos de guaroa',
    'san jose del fragua': 'san jose de fragua',
    'san pablo borbur': 'san pablo de borbur',
    'santacruz': 'santa cruz',
    'santiago de tolu': 'tolu',
    'tiquisio (pto. rico)': 'tiquisio',
    'villa de leyva': 'villa de leiva',
    'vistahermosa': 'vista hermosa',
    'yondo-casabe': 'yondo',

    # Casos truncados / incompletos
    'el canton del san pablo (man.': 'el canton del san pablo',
    'union panamericana (las animas': 'union panamericana'
}

# =========================================
# 2. Aplicar reemplazos
# =========================================

df_maestro['municipio'] = df_maestro['municipio'].replace(reemplazos_municipio)

# =========================================
# 3. Verificar resultado
# =========================================

valores_unicos_municipio = sorted(df_maestro['municipio'].dropna().unique().tolist())

print("Valores únicos en 'municipio' después de limpiar:")
for i, val in enumerate(valores_unicos_municipio, 1):
    print(f"{i}. {val}")

print(f"\nTotal de valores únicos: {len(valores_unicos_municipio)}")

Valores únicos en 'municipio' después de limpiar:
1. abejorral
2. abrego
3. abriaqui
4. acacias
5. acandi
6. acevedo
7. achi
8. agrado
9. agua de dios
10. aguachica
11. aguada
12. aguadas
13. aguazul
14. agustin codazzi
15. aipe
16. alban
17. albania
18. alcala
19. aldana
20. alejandria
21. alemania
22. algarrobo
23. algeciras
24. almaguer
25. almeida
26. alpujarra
27. altamira
28. alto baudo
29. altos del rosario
30. alvarado
31. amaga
32. amalfi
33. ambalema
34. anapoima
35. ancuya
36. andalucia
37. andes
38. angelopolis
39. angostura
40. anolaima
41. anori
42. anserma
43. ansermanuevo
44. antillas holandesas
45. antioquia
46. anza
47. anzoategui
48. apartado
49. apia
50. apulo
51. aquitania
52. aracataca
53. aranzazu
54. aratoca
55. arauca
56. arauquita
57. arbelaez
58. arboleda
59. arboledas
60. arboletes
61. arcabuco
62. arenal
63. argelia
64. argentina
65. ariguani
66. arjona
67. armenia
68. armero
69. arroyo hondo
70. aruba
71. arzerbaiyan
72. astrea
73. ataco
74. atrato
75. aus

In [69]:
# =========================================
# 1. Diccionario de reemplazos para 'corporacion'
# =========================================

reemplazos_corporacion = {
    'asamblea': 'asamblea departamental',
    'asamblea departamental': 'asamblea departamental',
    'concejo municipal': 'concejo',
    'concejo': 'concejo',
    'camara': 'camara',
    'camara de representantes': 'camara',
    'jal': 'jal',
    'senado': 'senado',
    'alcaldia municipal': 'alcaldia',
    'alcalde': 'alcaldia',
    'presidencia': 'presidencia',
    'gobernador': 'gobernacion',
    'consultas': 'consultas'
}

# =========================================
# 2. Aplicar reemplazos
# =========================================

df_maestro['corporacion'] = df_maestro['corporacion'].replace(reemplazos_corporacion)

# =========================================
# 3. Verificar resultado
# =========================================

valores_unicos_corporacion = df_maestro['corporacion'].dropna().unique().tolist()

print("Valores únicos en 'corporacion' después de limpiar:")
for i, val in enumerate(valores_unicos_corporacion, 1):
    print(f"{i}. {val}")

print(f"\nTotal de valores únicos: {len(valores_unicos_corporacion)}")

Valores únicos en 'corporacion' después de limpiar:
1. camara
2. presidencia
3. senado
4. asamblea departamental
5. concejo
6. alcaldia
7. jal
8. gobernacion
9. consultas

Total de valores únicos: 9


In [73]:
# Reemplazos directos en la columna 'candidato'
reemplazos_candidato = {
    'lista - mira': 'lista',
    'nuevo liberalismo - conservador -colombia justa li bres': 'lista',
    'partido nuevo liberalismo': 'lista'
}

df_maestro['candidato'] = df_maestro['candidato'].replace(reemplazos_candidato)

# Si quieres unificar cualquier valor que contenga 'lista' y una coalición adicional:
mask_lista = df_maestro['candidato'].str.contains(r'^lista\b', case=False, na=False)
df_maestro.loc[mask_lista, 'candidato'] = 'lista'

# Verificación
valores_unicos_candidato = sorted(df_maestro['candidato'].dropna().unique().tolist())
print("Total de valores únicos:", len(valores_unicos_candidato))
print(valores_unicos_candidato[:100])

Total de valores únicos: 4733
['abaned aldana lombana', 'abel alberto polanco lemos', 'abel farfan martinez', 'abel jose barrios martinez', 'abelardo duenez blanco', 'abelardo fabio morales alvarez', 'abelardo palacios gonzalez', 'abelino cardoso perdomo', 'abigail ramirez balaguera', 'abilio parada cote', 'abraham de jesus saker gonzalez', 'abraham ramirez hernandez', 'abundino rayo mahecha', 'acuerdo de coalicion partidos de la u y nuevo libe ralismo', 'adalberto lombana quintero', 'adalberto maldonado gonzalez', 'adalberto ovalle munoz', 'adalgisa rubis montero pabon', 'adela calderon arguello', 'adelaida estrella ruiz garcia', 'adelina isabel tejada mosquera', 'adelmo marulanda ruiz', 'ademir cardoso herrera', 'ader alonso anaya montesino', 'adinael contreras gonzalez', 'adolfo emilio ayala petro', 'adolfo yovanni vera', 'adonai morales ayazo', 'adrian norena gaviria', 'adrian selket jimenez lozada', 'adriana enith mora acosta', 'adriana feo zapata', 'adriana garcia garcia', 'adria

In [77]:
import pandas as pd

# Asegurar tipos consistentes
df_maestro['ano'] = pd.to_numeric(df_maestro['ano'], errors='coerce')
df_maestro['coddpto'] = df_maestro['coddpto'].astype(str).str.strip()
df_maestro['codmpio'] = df_maestro['codmpio'].astype(str).str.strip()
df_maestro['departamento'] = df_maestro['departamento'].astype(str).str.strip()
df_maestro['municipio'] = df_maestro['municipio'].astype(str).str.strip()

# Crear clave territorial
df_maestro['clave_territorial'] = (
    df_maestro['coddpto'] + '-' + df_maestro['codmpio']
)

# Ver cuántas combinaciones únicas hay por año
resumen_anual = (
    df_maestro
    .groupby('ano', dropna=False)
    .agg(
        registros=('clave_territorial', 'size'),
        claves_unicas=('clave_territorial', 'nunique'),
        deptos_unicos=('coddpto', 'nunique'),
        municipios_unicos=('codmpio', 'nunique')
    )
    .reset_index()
    .sort_values('ano')
)

print("Resumen por año:")
print(resumen_anual)

# Detectar claves territoriales que cambian de nombre según el año
cambios_territoriales = (
    df_maestro
    .groupby('clave_territorial')
    .agg(
        anios=('ano', lambda x: sorted(pd.Series(x).dropna().unique().tolist())),
        departamentos=('departamento', lambda x: sorted(pd.Series(x).dropna().unique().tolist())),
        municipios=('municipio', lambda x: sorted(pd.Series(x).dropna().unique().tolist()))
    )
    .reset_index()
)

# Quedarse con las claves que tienen más de una forma de nombre o aparecen en varios años
cambios_territoriales = cambios_territoriales[
    (cambios_territoriales['anios'].apply(len) > 1) |
    (cambios_territoriales['departamentos'].apply(len) > 1) |
    (cambios_territoriales['municipios'].apply(len) > 1)
].sort_values('clave_territorial')

print("\nClaves territoriales con cambios de nombre o presencia en varios años:")
print(cambios_territoriales.head(50))

# Exportar revisión
resumen_anual.to_csv('output/resumen_anual_codigos.csv', index=False)
cambios_territoriales.to_csv('output/cambios_territoriales.csv', index=False)

Resumen por año:
    ano  registros  claves_unicas  deptos_unicos  municipios_unicos
0  1982       1342           1062             34               1060
1  1986      10612           1057             27               1019
2  1988       2188            678             17                678
3  2022     188414           1165             34                304
4  2023     695629            628             31                212
5  2026     194179           1055             32                294

Claves territoriales con cambios de nombre o presencia en varios años:
   clave_territorial         anios departamentos       municipios
0                1-1  [2022, 2023]   [antioquia]       [medellin]
1               1-10  [2022, 2023]   [antioquia]     [alejandria]
3              1-103  [2022, 2023]   [antioquia]     [copacabana]
4              1-106  [2022, 2023]   [antioquia]      [chigorodo]
5              1-109  [2022, 2023]   [antioquia]        [dabeiba]
8              1-117  [2022, 2023]   [a

In [72]:
df_maestro_sql.to_csv(r"C:\Users\qluis\Documents\Proyectos\Nuevo liberalismo\output\df_maestro_candidatos.csv", index=False, encoding="utf-8-sig")

In [70]:
# Obtener valores únicos de la columna 'candidato'
valores_unicos_candidato = sorted(df_maestro['candidato'].dropna().unique().tolist())

print("Valores únicos en 'candidato':")
for i, val in enumerate(valores_unicos_candidato, 1):
    print(f"{i}. {val}")

print(f"\nTotal de valores únicos: {len(valores_unicos_candidato)}")

Valores únicos en 'candidato':
1. abaned aldana lombana
2. abel alberto polanco lemos
3. abel farfan martinez
4. abel jose barrios martinez
5. abelardo duenez blanco
6. abelardo fabio morales alvarez
7. abelardo palacios gonzalez
8. abelino cardoso perdomo
9. abigail ramirez balaguera
10. abilio parada cote
11. abraham de jesus saker gonzalez
12. abraham ramirez hernandez
13. abundino rayo mahecha
14. acuerdo de coalicion partidos de la u y nuevo libe ralismo
15. adalberto lombana quintero
16. adalberto maldonado gonzalez
17. adalberto ovalle munoz
18. adalgisa rubis montero pabon
19. adela calderon arguello
20. adelaida estrella ruiz garcia
21. adelina isabel tejada mosquera
22. adelmo marulanda ruiz
23. ademir cardoso herrera
24. ader alonso anaya montesino
25. adinael contreras gonzalez
26. adolfo emilio ayala petro
27. adolfo yovanni vera
28. adonai morales ayazo
29. adrian norena gaviria
30. adrian selket jimenez lozada
31. adriana enith mora acosta
32. adriana feo zapata
33. adri

In [76]:
import duckdb

query_sql = """
SELECT DISTINCT (codmpio)
FROM 
    df_maestro

"""

# Ejecutar la consulta y convertir el resultado a un nuevo DataFrame
df_maestro_sql = duckdb.sql(query_sql).df()
df_maestro_sql.head(100)

,codmpio
0,50318
1,5040
2,5044
3,5051
4,5059
...,...
95,198
96,17
97,360
98,135


In [78]:
import pandas as pd

# Normalizar tipos
for col in ['departamento', 'municipio', 'coddpto', 'codmpio']:
    df_maestro[col] = df_maestro[col].astype(str).str.strip()

# Quitar posibles nulos codificados como 'nan'
df_chk = df_maestro.copy()
df_chk = df_chk[~df_chk['departamento'].isin(['nan', 'None', ''])].copy()
df_chk = df_chk[~df_chk['municipio'].isin(['nan', 'None', ''])].copy()
df_chk = df_chk[~df_chk['coddpto'].isin(['nan', 'None', ''])].copy()
df_chk = df_chk[~df_chk['codmpio'].isin(['nan', 'None', ''])].copy()

# 1) Para cada (departamento, municipio), cuántos códigos distintos hay
codigos_por_nombre = (
    df_chk.groupby(['departamento', 'municipio'], dropna=False)
    .agg(
        coddpto_unicos=('coddpto', 'nunique'),
        codmpio_unicos=('codmpio', 'nunique'),
        registros=('municipio', 'size')
    )
    .reset_index()
)

# Casos problemáticos: más de un código para el mismo nombre
problemas_nombre = codigos_por_nombre[
    (codigos_por_nombre['coddpto_unicos'] > 1) |
    (codigos_por_nombre['codmpio_unicos'] > 1)
].sort_values(['departamento', 'municipio'])

print("Casos donde un mismo nombre tiene más de un código:")
print(problemas_nombre.head(200))
print("\nTotal de casos problemáticos:", len(problemas_nombre))

# 2) Para cada código, cuántos nombres distintos hay
nombres_por_codigo = (
    df_chk.groupby(['coddpto', 'codmpio'], dropna=False)
    .agg(
        departamentos_unicos=('departamento', 'nunique'),
        municipios_unicos=('municipio', 'nunique'),
        registros=('municipio', 'size')
    )
    .reset_index()
)

# Casos problemáticos: un mismo código asociado a más de un nombre
problemas_codigo = nombres_por_codigo[
    (nombres_por_codigo['departamentos_unicos'] > 1) |
    (nombres_por_codigo['municipios_unicos'] > 1)
].sort_values(['coddpto', 'codmpio'])

print("\nCasos donde un mismo código tiene más de un nombre:")
print(problemas_codigo.head(200))
print("\nTotal de casos problemáticos por código:", len(problemas_codigo))

# Exportar resultados
problemas_nombre.to_csv('output/problemas_nombre_vs_codigo.csv', index=False)
problemas_codigo.to_csv('output/problemas_codigo_vs_nombre.csv', index=False)
codigos_por_nombre.to_csv('output/codigos_por_nombre.csv', index=False)
nombres_por_codigo.to_csv('output/nombres_por_codigo.csv', index=False)

Casos donde un mismo nombre tiene más de un código:
    departamento      municipio  coddpto_unicos  codmpio_unicos  registros
0       amazonas     el encanto               2               2          7
1       amazonas    la chorrera               2               2          4
4       amazonas        leticia               2               2        289
8       amazonas  puerto narino               2               2         32
11     antioquia      abejorral               2               2         36
..           ...            ...             ...             ...        ...
237       boyaca        briceno               2               2         69
238       boyaca     buenavista               2               2        152
239       boyaca       busbanza               2               2         40
240       boyaca         caldas               2               2        207
241       boyaca   campohermoso               2               2         95

[200 rows x 5 columns]

Total de casos problemá

In [79]:
import pandas as pd
import numpy as np
import unicodedata
import re

# df_maestro: dataframe original
df = df_maestro.copy()

def norm_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8')
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

# Normalización básica
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
for col in ['departamento', 'municipio', 'coddpto', 'codmpio']:
    df[col] = df[col].astype(str).str.strip()

df['departamento_norm'] = df['departamento'].map(norm_text)
df['municipio_norm'] = df['municipio'].map(norm_text)

# Filtrar filas utilizables
valid = df[
    df['departamento_norm'].notna() &
    df['municipio_norm'].notna() &
    ~df['departamento_norm'].isin(['', 'nan', 'none']) &
    ~df['municipio_norm'].isin(['', 'nan', 'none']) &
    ~df['coddpto'].isin(['', 'nan', 'None', 'none']) &
    ~df['codmpio'].isin(['', 'nan', 'None', 'none'])
].copy()

# Prioridad a años recientes
prioridad_anios = {2026: 6, 2023: 5, 2022: 4, 1988: 3, 1986: 2, 1982: 1}
valid['prioridad'] = valid['ano'].map(prioridad_anios).fillna(0).astype(int)

# Conteo por combinación nombre->código
resumen = (
    valid.groupby(
        ['departamento_norm', 'municipio_norm', 'coddpto', 'codmpio', 'ano'],
        dropna=False
    )
    .size()
    .reset_index(name='registros')
)

resumen['prioridad'] = resumen['ano'].map(prioridad_anios).fillna(0).astype(int)

# Puntaje para escoger código canónico: año más reciente primero, luego mayor frecuencia
canon = (
    resumen.sort_values(
        ['departamento_norm', 'municipio_norm', 'prioridad', 'registros'],
        ascending=[True, True, False, False]
    )
    .groupby(['departamento_norm', 'municipio_norm'], as_index=False)
    .first()
)

canon = canon.rename(columns={
    'coddpto': 'coddpto_canon',
    'codmpio': 'codmpio_canon',
    'ano': 'ano_referencia',
    'registros': 'registros_referencia'
})

# Pegar código canónico al dataset completo
df2 = df.merge(
    canon[['departamento_norm', 'municipio_norm', 'coddpto_canon', 'codmpio_canon', 'ano_referencia']],
    on=['departamento_norm', 'municipio_norm'],
    how='left'
)

# Reemplazar con código canónico donde exista
df2['coddpto_original'] = df2['coddpto']
df2['codmpio_original'] = df2['codmpio']

df2['coddpto'] = df2['coddpto_canon'].combine_first(df2['coddpto'])
df2['codmpio'] = df2['codmpio_canon'].combine_first(df2['codmpio'])

# Crear clave territorial final
df2['clave_territorial'] = df2['coddpto'].astype(str).str.strip() + '-' + df2['codmpio'].astype(str).str.strip()

# Diccionario maestro por código final para regenerar nombres uniformes
nombres_codigo = (
    df2[
        df2['coddpto'].notna() & df2['codmpio'].notna() &
        df2['departamento_norm'].notna() & df2['municipio_norm'].notna()
    ]
    .groupby(['coddpto', 'codmpio', 'departamento_norm', 'municipio_norm'])
    .size()
    .reset_index(name='registros')
)

# Priorizar nombres asociados a años recientes
tmp = df2[['coddpto', 'codmpio', 'departamento', 'municipio', 'departamento_norm', 'municipio_norm', 'ano']].copy()
tmp['prioridad'] = tmp['ano'].map(prioridad_anios).fillna(0).astype(int)

nombres_preferidos = (
    tmp.groupby(['coddpto', 'codmpio', 'departamento', 'municipio', 'departamento_norm', 'municipio_norm', 'prioridad'])
    .size()
    .reset_index(name='registros')
    .sort_values(['coddpto', 'codmpio', 'prioridad', 'registros'], ascending=[True, True, False, False])
    .groupby(['coddpto', 'codmpio'], as_index=False)
    .first()
)

nombres_preferidos = nombres_preferidos.rename(columns={
    'departamento': 'departamento_canon',
    'municipio': 'municipio_canon'
})

df2 = df2.merge(
    nombres_preferidos[['coddpto', 'codmpio', 'departamento_canon', 'municipio_canon']],
    on=['coddpto', 'codmpio'],
    how='left'
)

# Uniformar nombres según código final
df2['departamento'] = df2['departamento_canon'].combine_first(df2['departamento'])
df2['municipio'] = df2['municipio_canon'].combine_first(df2['municipio'])

# Auditoría: nombre->código y código->nombre después del ajuste
audit_nombre = (
    df2.groupby(['departamento', 'municipio'])
    .agg(
        coddpto_unicos=('coddpto', 'nunique'),
        codmpio_unicos=('codmpio', 'nunique'),
        registros=('municipio', 'size')
    )
    .reset_index()
)

audit_nombre_conf = audit_nombre[
    (audit_nombre['coddpto_unicos'] > 1) | (audit_nombre['codmpio_unicos'] > 1)
].sort_values(['departamento', 'municipio'])

audit_codigo = (
    df2.groupby(['coddpto', 'codmpio'])
    .agg(
        departamentos_unicos=('departamento', 'nunique'),
        municipios_unicos=('municipio', 'nunique'),
        registros=('municipio', 'size')
    )
    .reset_index()
)

audit_codigo_conf = audit_codigo[
    (audit_codigo['departamentos_unicos'] > 1) | (audit_codigo['municipios_unicos'] > 1)
].sort_values(['coddpto', 'codmpio'])

# Trazabilidad de cambios
trazabilidad = df2[
    (df2['coddpto_original'] != df2['coddpto']) |
    (df2['codmpio_original'] != df2['codmpio'])
].copy()

# Exportables
canon.to_csv('output/diccionario_codigos_canonicos.csv', index=False)
trazabilidad.to_csv('output/trazabilidad_cambios_codigos.csv', index=False)
audit_nombre_conf.to_csv('output/auditoria_nombre_a_codigo_postajuste.csv', index=False)
audit_codigo_conf.to_csv('output/auditoria_codigo_a_nombre_postajuste.csv', index=False)
df2.to_csv('output/dataset_homologado.csv', index=False)

In [80]:
df2.head()

,ano,corporacion,coddpto,departamento,codmpio,municipio,votos,candidato,partido,clave_territorial,departamento_norm,municipio_norm,coddpto_canon,codmpio_canon,ano_referencia,coddpto_original,codmpio_original,departamento_canon,municipio_canon
0,1982,camara,11,cauca,1,popayan,1411,lista,nuevo liberalismo,11-1,cauca,popayan,11,1,2026,19,19001,cauca,popayan
1,1982,camara,11,cauca,1,popayan,0,omar henry velasco ramirez,nuevo liberalismo,11-1,cauca,popayan,11,1,2026,19,19001,cauca,popayan
2,1982,camara,11,cauca,4,almaguer,5,lista,nuevo liberalismo,11-4,cauca,almaguer,11,4,2026,19,19022,cauca,almaguer
3,1982,camara,11,cauca,4,almaguer,0,omar henry velasco ramirez,nuevo liberalismo,11-4,cauca,almaguer,11,4,2026,19,19022,cauca,almaguer
4,1982,camara,11,cauca,5,argelia,108,lista,nuevo liberalismo,11-5,cauca,argelia,11,5,2026,19,19050,cauca,argelia


In [83]:
df2.columns

Index(['ano', 'corporacion', 'coddpto', 'departamento', 'codmpio', 'municipio',
       'votos', 'candidato', 'partido', 'clave_territorial',
       'departamento_norm', 'municipio_norm', 'coddpto_canon', 'codmpio_canon',
       'ano_referencia', 'coddpto_original', 'codmpio_original',
       'departamento_canon', 'municipio_canon'],
      dtype='object')

In [ ]:
df2.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1092364 entries, 0 to 1092363
Data columns (total 19 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   ano                 1092364 non-null  int64 
 1   corporacion         1092364 non-null  object
 2   coddpto             1092364 non-null  object
 3   departamento        1092364 non-null  object
 4   codmpio             1092364 non-null  object
 5   municipio           1092364 non-null  object
 6   votos               1092364 non-null  int64 
 7   candidato           1092364 non-null  object
 8   partido             1092364 non-null  object
 9   clave_territorial   1092364 non-null  object
 10  departamento_norm   1092364 non-null  object
 11  municipio_norm      1092364 non-null  object
 12  coddpto_canon       1092364 non-null  object
 13  codmpio_canon       1092364 non-null  object
 14  ano_referencia      1092364 non-null  int64 
 15  coddpto_original    1092364 non-

In [84]:
cols_finales = [
    'ano', 'corporacion', 'coddpto', 'departamento',
    'codmpio', 'municipio', 'votos', 'candidato',
    'partido', 'clave_territorial', 'coddpto_canon', 'codmpio_canon'
]

df_final = df2[cols_finales].copy()
df_final.to_parquet('output/dataset_final.parquet', index=False)

In [86]:
df_final.to_csv('output/dataset_final.csv', index=False, encoding='utf-8-sig')